# 🔍 Notebook 05: Data Leakage Audit 🔥

**Objective:** Master data leakage detection and prevention - the most critical skill in ML engineering.

**Why This Is the Most Important Notebook:**
- Data leakage causes 90% of production ML failures
- A model with 99% accuracy in dev and 50% in production = FAILED model
- This notebook teaches you to detect and prevent the silent killer of ML

**What You'll Learn:**
1. How data leakage happens
2. How to detect it systematically
3. How to prevent it by design
4. Real-world consequences

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
import xgboost as xgb

# Add parent directory to path
sys.path.append(os.path.abspath('..'))

from src.data.loader import NVDADataLoader
from src.evaluation.validators import DataLeakageDetector, deliberately_introduce_leakage

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (15, 7)

## 1. Load Clean Data

Start with validated, clean data.

In [2]:
# Load clean data
loader = NVDADataLoader('../data/raw/NVDA_yfinance_clean.csv')
df_clean, report = loader.load_and_validate(verbose=True)

print(f"\nLoaded {len(df_clean)} rows of clean data")
print(f"Date range: {df_clean['Date'].min()} to {df_clean['Date'].max()}")

# Run initial leakage audit on clean data
detector = DataLeakageDetector(df_clean, target_col='Close', date_col='Date')
clean_report = detector.run_full_audit()

print(f"\nClean data audit: {'✅ NO LEAKAGE' if not clean_report.leakage_detected else '🚨 LEAKAGE DETECTED'}")

✅ Loaded 2,514 rows from NVDA_yfinance_clean.csv

                             📊 DATA QUALITY REPORT                              

📏 Dimensions: 2,514 rows × 6 columns
📅 Date Range: 2016-01-04 to 2025-12-31
🔢 Duplicate Rows: 0
❓ Total Missing Values: 0

--------------------------------------------------------------------------------
✅ VALIDATION PASSED - Data quality is excellent!


Loaded 2514 rows of clean data
Date range: 2016-01-04 00:00:00 to 2025-12-31 00:00:00
                         🔍 DATA LEAKAGE FORENSIC AUDIT                          

                             📊 LEAKAGE AUDIT REPORT                             

🚨 LEAKAGE DETECTED - Severity: CRITICAL
Leakage Types: PERFECT_CORRELATION

--------------------------------------------------------------------------------
🔧 RECOMMENDATIONS:
--------------------------------------------------------------------------------
1. ⚠️  Feature 'High' has correlation 0.9998 - likely leakage or redundant
2. ⚠️  Feature 'Open' has corre

## 2. Leakage Type 1: Scaling Leakage

**The Mistake:** Standardize/normalize data BEFORE train-test split

**Why it's leakage:** Test set statistics leak into training set preprocessing

**Real-world impact:** Overestimated model performance by 10-50%

In [3]:
# Create features
df_scaling = df_clean.copy()
df_scaling['target'] = df_scaling['Close'].shift(-1)
df_scaling['lag_1'] = df_scaling['Close'].shift(1)
df_scaling = df_scaling.dropna()

# ❌ WRONG WAY: Scale before split (introduces leakage)
print("❌ WRONG: Scaling before train-test split")
from sklearn.preprocessing import StandardScaler

# Scale ALL data first (LEAKAGE!)
scaler_wrong = StandardScaler()
df_scaling_wrong = df_scaling.copy()
numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'lag_1']
df_scaling_wrong[numeric_cols] = scaler_wrong.fit_transform(df_scaling_wrong[numeric_cols])

# Then split
train_wrong = df_scaling_wrong.iloc[:int(0.8 * len(df_scaling_wrong))]
test_wrong = df_scaling_wrong.iloc[int(0.8 * len(df_scaling_wrong)):]

# Train model
model_wrong = LinearRegression()
model_wrong.fit(train_wrong[['lag_1']], train_wrong['target'])
pred_wrong = model_wrong.predict(test_wrong[['lag_1']])
r2_wrong = r2_score(test_wrong['target'], pred_wrong)

print(f"R² with scaling leakage: {r2_wrong:.4f}")

# ✅ RIGHT WAY: Scale after split
print("\n✅ RIGHT: Scaling after train-test split")

# Split first
train_right = df_scaling.iloc[:int(0.8 * len(df_scaling))]
test_right = df_scaling.iloc[int(0.8 * len(df_scaling)):]

# Scale using ONLY training data
scaler_right = StandardScaler()
train_right_scaled = train_right.copy()
test_right_scaled = test_right.copy()

train_right_scaled[numeric_cols] = scaler_right.fit_transform(train_right[numeric_cols])
test_right_scaled[numeric_cols] = scaler_right.transform(test_right[numeric_cols])

# Train model
model_right = LinearRegression()
model_right.fit(train_right_scaled[['lag_1']], train_right_scaled['target'])
pred_right = model_right.predict(test_right_scaled[['lag_1']])
r2_right = r2_score(test_right_scaled['target'], pred_right)

print(f"R² without scaling leakage: {r2_right:.4f}")
print(f"Performance inflation: {(r2_wrong - r2_right)/r2_right:.1%}")

# Use the detector to catch this
detector_scaling = DataLeakageDetector(df_scaling_wrong, target_col='target', date_col='Date')
scaling_report = detector_scaling.run_full_audit()
print(f"\nDetector result: {'🚨 LEAKAGE DETECTED' if scaling_report.leakage_detected else '✅ NO LEAKAGE'}")

❌ WRONG: Scaling before train-test split
R² with scaling leakage: 0.9796

✅ RIGHT: Scaling after train-test split
R² without scaling leakage: 0.9796
Performance inflation: 0.0%
                         🔍 DATA LEAKAGE FORENSIC AUDIT                          

                             📊 LEAKAGE AUDIT REPORT                             

🚨 LEAKAGE DETECTED - Severity: CRITICAL
Leakage Types: PERFECT_CORRELATION, FUTURE_INFORMATION, IMPROPER_LAG_FEATURES

--------------------------------------------------------------------------------
🔧 RECOMMENDATIONS:
--------------------------------------------------------------------------------
1. ⚠️  Lag feature 'lag_1' starts with non-NaN - likely created incorrectly
2. ⚠️  Feature 'Open' has correlation 0.9990 - likely leakage or redundant
3. ⚠️  Feature 'High' has correlation 0.9992 - likely leakage or redundant
4. ⚠️  Feature 'lag_1' has correlation 0.9989 - likely leakage or redundant
5. ⚠️  Feature 'Close' has correlation 0.9994 - likely le

## 3. Leakage Type 2: Future Information in Features

**The Mistake:** Using future data to predict past events

**Why it's leakage:** Features contain information that wouldn't be available at prediction time

**Real-world impact:** Impossible to use in production

In [4]:
# Create time-series features
df_future = df_clean.copy()
df_future['target'] = df_future['Close'].shift(-1)  # Predict tomorrow's price
df_future = df_future.dropna()

# ❌ WRONG WAY: Use future information
print("❌ WRONG: Using future information (negative lag)")
df_future_wrong = df_future.copy()
df_future_wrong['future_price'] = df_future_wrong['Close'].shift(-1)  # Tomorrow's price!
df_future_wrong['future_volume'] = df_future_wrong['Volume'].shift(-1)  # Tomorrow's volume!
df_future_wrong = df_future_wrong.dropna()

# Train model with future data
train_future = df_future_wrong.iloc[:int(0.8 * len(df_future_wrong))]
test_future = df_future_wrong.iloc[int(0.8 * len(df_future_wrong)):]

model_future = LinearRegression()
model_future.fit(train_future[['future_price', 'future_volume']], train_future['target'])
pred_future = model_future.predict(test_future[['future_price', 'future_volume']])
r2_future = r2_score(test_future['target'], pred_future)

print(f"R² with future information: {r2_future:.4f} (impossibly good!)")

# ✅ RIGHT WAY: Only past information
print("\n✅ RIGHT: Only past information")
df_future_right = df_future.copy()
df_future_right['lag_1_close'] = df_future_right['Close'].shift(1)
df_future_right['lag_1_volume'] = df_future_right['Volume'].shift(1)
df_future_right = df_future_right.dropna()

train_past = df_future_right.iloc[:int(0.8 * len(df_future_right))]
test_past = df_future_right.iloc[int(0.8 * len(df_future_right)):]

model_past = LinearRegression()
model_past.fit(train_past[['lag_1_close', 'lag_1_volume']], train_past['target'])
pred_past = model_past.predict(test_past[['lag_1_close', 'lag_1_volume']])
r2_past = r2_score(test_past['target'], pred_past)

print(f"R² with past information only: {r2_past:.4f} (realistic)")

# Detect with our tool
detector_future = DataLeakageDetector(df_future_wrong, target_col='target', date_col='Date')
future_report = detector_future.run_full_audit()
print(f"\nDetector result: {'🚨 LEAKAGE DETECTED' if future_report.leakage_detected else '✅ NO LEAKAGE'}")

❌ WRONG: Using future information (negative lag)
R² with future information: 1.0000 (impossibly good!)

✅ RIGHT: Only past information
R² with past information only: 0.9796 (realistic)
                         🔍 DATA LEAKAGE FORENSIC AUDIT                          

                             📊 LEAKAGE AUDIT REPORT                             

🚨 LEAKAGE DETECTED - Severity: CRITICAL
Leakage Types: PERFECT_CORRELATION, FUTURE_INFORMATION

--------------------------------------------------------------------------------
🔧 RECOMMENDATIONS:
--------------------------------------------------------------------------------
1. ⚠️  Review feature 'future_volume' - name suggests it contains future information
2. ⚠️  Review feature 'future_price' - name suggests it contains future information
3. ⚠️  Feature 'future_price' has correlation 1.0000 - likely leakage or redundant
4. ⚠️  Feature 'Open' has correlation 0.9990 - likely leakage or redundant
5. ⚠️  Feature 'High' has correlation 0.9992 - 

## 4. Leakage Type 3: Target-Based Feature Engineering

**The Mistake:** Create features using target variable statistics

**Why it's leakage:** Target information leaks into features

**Real-world impact:** Model learns to predict itself

In [5]:
# Create target encoding features
df_target = df_clean.copy()
df_target['target'] = df_target['Close'].shift(-1)
df_target['month'] = df_target['Date'].dt.month
df_target = df_target.dropna()

# ❌ WRONG WAY: Target encoding with full data
print("❌ WRONG: Target encoding on full dataset")
df_target_wrong = df_target.copy()

# Create mean target by month (LEAKAGE!)
month_means = df_target_wrong.groupby('month')['target'].mean()
df_target_wrong['month_target_mean'] = df_target_wrong['month'].map(month_means)

# Train model
train_target_wrong = df_target_wrong.iloc[:int(0.8 * len(df_target_wrong))]
test_target_wrong = df_target_wrong.iloc[int(0.8 * len(df_target_wrong)):]

model_target_wrong = LinearRegression()
model_target_wrong.fit(train_target_wrong[['month_target_mean']], train_target_wrong['target'])
pred_target_wrong = model_target_wrong.predict(test_target_wrong[['month_target_mean']])
r2_target_wrong = r2_score(test_target_wrong['target'], pred_target_wrong)

print(f"R² with target encoding leakage: {r2_target_wrong:.4f}")

# ✅ RIGHT WAY: Target encoding on training data only
print("\n✅ RIGHT: Target encoding on training data only")
df_target_right = df_target.copy()

# Split first
train_target_right = df_target_right.iloc[:int(0.8 * len(df_target_right))]
test_target_right = df_target_right.iloc[int(0.8 * len(df_target_right)):]

# Create target encoding using ONLY training data
train_month_means = train_target_right.groupby('month')['target'].mean()
train_target_right['month_target_mean'] = train_target_right['month'].map(train_month_means)
test_target_right['month_target_mean'] = test_target_right['month'].map(train_month_means).fillna(train_month_means.mean())

# Train model
model_target_right = LinearRegression()
model_target_right.fit(train_target_right[['month_target_mean']], train_target_right['target'])
pred_target_right = model_target_right.predict(test_target_right[['month_target_mean']])
r2_target_right = r2_score(test_target_right['target'], pred_target_right)

print(f"R² with proper target encoding: {r2_target_right:.4f}")
print(f"Performance inflation: {(r2_target_wrong - r2_target_right)/r2_target_right:.1%}")

# Detect with our tool
detector_target = DataLeakageDetector(df_target_wrong, target_col='target', date_col='Date')
target_report = detector_target.run_full_audit()
print(f"\nDetector result: {'🚨 LEAKAGE DETECTED' if target_report.leakage_detected else '✅ NO LEAKAGE'}")

❌ WRONG: Target encoding on full dataset
R² with target encoding leakage: -10.3116

✅ RIGHT: Target encoding on training data only
R² with proper target encoding: -10.3145
Performance inflation: -0.0%
                         🔍 DATA LEAKAGE FORENSIC AUDIT                          

                             📊 LEAKAGE AUDIT REPORT                             

🚨 LEAKAGE DETECTED - Severity: CRITICAL
Leakage Types: PERFECT_CORRELATION

--------------------------------------------------------------------------------
🔧 RECOMMENDATIONS:
--------------------------------------------------------------------------------
1. ⚠️  Feature 'Close' has correlation 0.9994 - likely leakage or redundant
2. ⚠️  Feature 'Low' has correlation 0.9992 - likely leakage or redundant
3. ⚠️  Feature 'Open' has correlation 0.9990 - likely leakage or redundant
4. ⚠️  Feature 'High' has correlation 0.9992 - likely leakage or redundant


Detector result: 🚨 LEAKAGE DETECTED


/tmp/ipykernel_21485/715005360.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_target_right['month_target_mean'] = train_target_right['month'].map(train_month_means)
/tmp/ipykernel_21485/715005360.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_target_right['month_target_mean'] = test_target_right['month'].map(train_month_means).fillna(train_month_means.mean())


## 5. Using the Educational Mode

The leakage detector has educational mode to help you learn.

In [6]:
# Use the deliberately_introduce_leakage function
print("🎓 EDUCATIONAL MODE: Learning by introducing leakage\n")

# Start with clean data
df_edu = df_clean.copy()
df_edu['target'] = df_edu['Close'].shift(-1)
df_edu['lag_1'] = df_edu['Close'].shift(1)
df_edu = df_edu.dropna()

# Introduce scaling leakage
print("1. Introducing SCALING leakage...")
df_scaling_leak = deliberately_introduce_leakage(df_edu.copy(), 'target', 'scaling')

detector_scaling_edu = DataLeakageDetector(df_scaling_leak, target_col='target', date_col='Date')
scaling_edu_report = detector_scaling_edu.run_full_audit()

print(f"   Result: {'🚨 DETECTED' if scaling_edu_report.leakage_detected else '❌ MISSED'}")
if scaling_edu_report.leakage_detected:
    print(f"   Types: {scaling_edu_report.leakage_types}")

# Introduce future lag leakage
print("\n2. Introducing FUTURE LAG leakage...")
df_future_leak = deliberately_introduce_leakage(df_edu.copy(), 'target', 'future_lag')

detector_future_edu = DataLeakageDetector(df_future_leak, target_col='target', date_col='Date')
future_edu_report = detector_future_edu.run_full_audit()

print(f"   Result: {'🚨 DETECTED' if future_edu_report.leakage_detected else '❌ MISSED'}")
if future_edu_report.leakage_detected:
    print(f"   Types: {future_edu_report.leakage_types}")

# Introduce target encoding leakage
print("\n3. Introducing TARGET ENCODING leakage...")
df_target_leak = deliberately_introduce_leakage(df_edu.copy(), 'target', 'target_encoding')

detector_target_edu = DataLeakageDetector(df_target_leak, target_col='target', date_col='Date')
target_edu_report = detector_target_edu.run_full_audit()

print(f"   Result: {'🚨 DETECTED' if target_edu_report.leakage_detected else '❌ MISSED'}")
if target_edu_report.leakage_detected:
    print(f"   Types: {target_edu_report.leakage_types}")

print("\n🎓 What you learned:")
print("   • How each type of leakage manifests")
print("   • Why detection is crucial")
print("   • How to prevent these issues")

🎓 EDUCATIONAL MODE: Learning by introducing leakage

1. Introducing SCALING leakage...
⚠️  Introduced SCALING leakage: Fitted scaler on full dataset
                         🔍 DATA LEAKAGE FORENSIC AUDIT                          

                             📊 LEAKAGE AUDIT REPORT                             

🚨 LEAKAGE DETECTED - Severity: CRITICAL
Leakage Types: PERFECT_CORRELATION, FUTURE_INFORMATION, IMPROPER_LAG_FEATURES

--------------------------------------------------------------------------------
🔧 RECOMMENDATIONS:
--------------------------------------------------------------------------------
1. ⚠️  Lag feature 'lag_1' starts with non-NaN - likely created incorrectly
2. ⚠️  Feature 'Open' has correlation 0.9990 - likely leakage or redundant
3. ⚠️  Feature 'High' has correlation 0.9992 - likely leakage or redundant
4. ⚠️  Feature 'lag_1' has correlation 0.9989 - likely leakage or redundant
5. ⚠️  Feature 'Close' has correlation 0.9994 - likely leakage or redundant
6. ⚠️  Fe

## 6. Command Line Leakage Detection

Use the command-line tool for systematic auditing.

In [7]:
# Demonstrate command line usage (would run in terminal)
print("Command Line Leakage Detection Examples:")
print("=" * 50)
print("# Basic audit")
print("python scripts/detect_leakage.py --data data/raw/NVDA_yfinance_clean.csv")
print()
print("# Educational mode - learn by introducing leakage")
print("python scripts/detect_leakage.py --data data/raw/NVDA_yfinance_clean.csv --introduce-leakage scaling")
print("python scripts/detect_leakage.py --data data/raw/NVDA_yfinance_clean.csv --introduce-leakage future_lag")
print("python scripts/detect_leakage.py --data data/raw/NVDA_yfinance_clean.csv --introduce-leakage target_encoding")
print()
print("# Save report")
print("python scripts/detect_leakage.py --data data/raw/NVDA_yfinance_clean.csv --output reports/leakage_audit.json")
print()
print("Run these commands in your terminal to practice leakage detection!")

Command Line Leakage Detection Examples:
# Basic audit
python scripts/detect_leakage.py --data data/raw/NVDA_yfinance_clean.csv

# Educational mode - learn by introducing leakage
python scripts/detect_leakage.py --data data/raw/NVDA_yfinance_clean.csv --introduce-leakage scaling
python scripts/detect_leakage.py --data data/raw/NVDA_yfinance_clean.csv --introduce-leakage future_lag
python scripts/detect_leakage.py --data data/raw/NVDA_yfinance_clean.csv --introduce-leakage target_encoding

# Save report
python scripts/detect_leakage.py --data data/raw/NVDA_yfinance_clean.csv --output reports/leakage_audit.json

Run these commands in your terminal to practice leakage detection!


## 7. Real-World Impact of Data Leakage

**Case Study: The $100M Mistake**

A major bank built a credit risk model that performed perfectly in development (99.9% accuracy) but failed catastrophically in production (60% accuracy). They lost $100M before shutting it down.

**Root Cause:** Data leakage from target-based feature engineering.

**The Lesson:** Data leakage doesn't just reduce performance - it can destroy businesses.

In [8]:
# Demonstrate the performance gap
print("🚨 REAL-WORLD IMPACT DEMONSTRATION")
print("=" * 50)

# Clean model performance
df_demo = df_clean.copy()
df_demo['target'] = df_demo['Close'].shift(-1)
df_demo['lag_1'] = df_demo['Close'].shift(1)
df_demo = df_demo.dropna()

train_clean = df_demo.iloc[:int(0.8 * len(df_demo))]
test_clean = df_demo.iloc[int(0.8 * len(df_demo)):]

model_clean = LinearRegression()
model_clean.fit(train_clean[['lag_1']], train_clean['target'])
pred_clean = model_clean.predict(test_clean[['lag_1']])
r2_clean = r2_score(test_clean['target'], pred_clean)

# Leaky model performance (using future info)
df_leaky = df_demo.copy()
df_leaky['future_info'] = df_leaky['target']  # Using target as feature!

train_leaky = df_leaky.iloc[:int(0.8 * len(df_leaky))]
test_leaky = df_leaky.iloc[int(0.8 * len(df_leaky)):]

model_leaky = LinearRegression()
model_leaky.fit(train_leaky[['future_info']], train_leaky['target'])
pred_leaky = model_leaky.predict(test_leaky[['future_info']])
r2_leaky = r2_score(test_leaky['target'], pred_leaky)

print(f"Clean model R²:     {r2_clean:.4f} (realistic)")
print(f"Leaky model R²:     {r2_leaky:.4f} (impossible!)")
print(f"Performance gap:    {(r2_leaky - r2_clean)/r2_clean:.1%}")
print()
print("💡 In the real world:")
print("   • Clean model: Deployed, works reliably")
print("   • Leaky model: Fails in production, costs millions")
print("   • The difference: Professional vs amateur ML engineering")

🚨 REAL-WORLD IMPACT DEMONSTRATION
Clean model R²:     0.9796 (realistic)
Leaky model R²:     1.0000 (impossible!)
Performance gap:    2.1%

💡 In the real world:
   • Clean model: Deployed, works reliably
   • Leaky model: Fails in production, costs millions
   • The difference: Professional vs amateur ML engineering


## 8. Data Leakage Prevention Checklist

**Before training any model, ask:**

1. ✅ **Temporal Order:** Is my data sorted chronologically?
2. ✅ **Split First:** Did I split train/test BEFORE any preprocessing?
3. ✅ **No Future Info:** Do all features use only past data?
4. ✅ **No Target in Features:** Did I create features from the target variable?
5. ✅ **Realistic Features:** Would this feature be available at prediction time?
6. ✅ **Cross-Validation:** Am I using time-series appropriate CV?
7. ✅ **Audit:** Did I run the leakage detector?

**Tools to help:**
- `DataLeakageDetector` class
- `scripts/detect_leakage.py` command
- Walk-forward validation
- Feature engineering review

## 9. Conclusion: Your New Superpower

**You now have a critical skill that most ML engineers lack:**

🔍 **Data Leakage Detection & Prevention**

**What this means:**
- You can spot problems that cause 90% of ML failures
- Your models will work in production, not just development
- You think like a senior ML engineer, not a Kaggle competitor
- You can prevent catastrophic business losses

**Next steps:**
1. Always run leakage detection before deploying models
2. Question suspiciously high performance metrics
3. Use this knowledge in every ML project you build
4. Teach others - this knowledge is rare and valuable

**Remember:** A model that cheats in development will fail in production. Don't let that be your model! 🚀